# Options from Zero · Episode 15: Delta, gamma, theta

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. Three ways to get the Greeks · 4. Delta across strikes · 5. Gamma · 6. Theta · 7. The trade-off · 8. The break-even move · 9. This episode's question · 10. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 15,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

In [ ]:
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": r * 100, "vol_pct": vol * 100}

Part 2's additions: Greeks, simulation, hedging, implied volatility, Black and Bachelier (`episodes/_shared/options_part2.py`):

In [ ]:
# Part 2 additions to options.py: the Greeks, simulation, a hedging simulator, implied volatility, and Black's and
# Bachelier's formulas on a forward (for FX and interest-rate options). Written out by hand and copied verbatim into
# each Part 2 notebook; the notebooks check them against QuantLib. Needs options.py (norm_cdf, black_scholes),
# math and numpy as np.

def norm_pdf(x):
    """The bell curve's height at x (standard normal density)."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def norm_cdf_v(x):
    """norm_cdf for a whole numpy array at once."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x, dtype=float) / math.sqrt(2)))

def bs_greeks(s, k, t, r, vol):
    """Price and Greeks of a European call on a share with no dividends.
    delta per $1 of share price, gamma per $1 squared, vega per 1.00 of volatility (100 vol points),
    theta per year (negative: value lost as time passes), rho per 1.00 of interest rate."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    disc = math.exp(-r * t)
    return {"price": s * norm_cdf(d1) - k * disc * norm_cdf(d2),
            "delta": norm_cdf(d1),
            "gamma": norm_pdf(d1) / (s * vol * math.sqrt(t)),
            "vega": s * norm_pdf(d1) * math.sqrt(t),
            "theta": -s * norm_pdf(d1) * vol / (2 * math.sqrt(t)) - r * k * disc * norm_cdf(d2),
            "rho": k * t * disc * norm_cdf(d2)}

def call_delta(s, k, t, r, vol):
    """Black-Scholes delta N(d1) for an array of share prices."""
    d1 = (np.log(np.asarray(s, dtype=float) / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    return norm_cdf_v(d1)

def simulate_paths(s, t, drift, vol, steps, n_paths, rng):
    """Share price paths: shape (n_paths, steps + 1), starting at s, lognormal steps with the given drift."""
    dt = t / steps
    z = rng.standard_normal((n_paths, steps))
    log_steps = (drift - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * z
    return s * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_steps, axis=1)], axis=1))

def monte_carlo_call(s, k, t, r, vol, n, rng):
    """Price = the average discounted payoff over n risk-neutral outcomes, with its standard error."""
    z = rng.standard_normal(n)
    s_t = s * np.exp((r - vol ** 2 / 2) * t + vol * math.sqrt(t) * z)
    pv = math.exp(-r * t) * np.maximum(s_t - k, 0.0)
    return float(pv.mean()), float(pv.std(ddof=1) / math.sqrt(n))

def hedge_short_call(s, k, t, r, vol_implied, vol_realised, steps, n_paths, rng, hedge=True):
    """Sell the call at vol_implied and, if hedge, rebalance to delta shares (computed at vol_implied) `steps` times.
    The share moves with vol_realised and grows at r. Cash earns r. Returns each path's P&L at expiry in today's dollars."""
    dt = t / steps
    price = s * np.ones(n_paths)
    held = call_delta(price, k, t, r, vol_implied) if hedge else np.zeros(n_paths)
    cash = black_scholes(s, k, t, r, vol_implied) - held * price      # premium received, shares bought
    for i in range(1, steps + 1):
        price = price * np.exp((r - vol_realised ** 2 / 2) * dt + vol_realised * math.sqrt(dt) * rng.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        if hedge and i < steps:
            new = call_delta(price, k, t - i * dt, r, vol_implied)
            cash = cash - (new - held) * price                        # buy (or sell) the difference
            held = new
    pnl = cash + held * price - np.maximum(price - k, 0.0)           # sell the shares, pay the option's payoff
    return pnl * math.exp(-r * t)

def implied_vol(price, s, k, t, r, lo=0.0001, hi=3.0, tol=1e-12):
    """Run Black-Scholes backwards by bisection: halve [lo, hi] until the price fits. Returns (vol, halvings)."""
    n = 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if black_scholes(s, k, t, r, mid) < price:
            lo = mid
        else:
            hi = mid
        n += 1
    return (lo + hi) / 2, n

def black76(forward, strike, t, vol, df, call=True):
    """Black's formula on a forward: df x [F N(d1) - K N(d2)] for a call. FX options (Garman-Kohlhagen) and caplets."""
    sd = vol * math.sqrt(t)
    d1 = (math.log(forward / strike) + sd ** 2 / 2) / sd
    d2 = d1 - sd
    if call:
        return df * (forward * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - forward * norm_cdf(-d1))

def bachelier(forward, strike, t, normal_vol, df, call=True):
    """Bachelier's formula: the forward moves by normal_vol x sqrt(t) in absolute terms (it can go below zero)."""
    sd = normal_vol * math.sqrt(t)
    d = (forward - strike) / sd
    if call:
        return df * ((forward - strike) * norm_cdf(d) + sd * norm_pdf(d))
    return df * ((strike - forward) * norm_cdf(-d) + sd * norm_pdf(d))

## 2. Last episode's question

Why wasn't the hedged spread exactly zero? We set the hedge once and left it for a whole day; the argument needs it adjusted continuously. Re-running Episode 14's test with the hedge held for an hour instead of a day, the risk left over shrinks about fivefold.

In [ ]:
rng = np.random.default_rng(15)
g0 = bs_greeks(S, K, T, r, vol)
left = {}
for label, dt_ in (("day", 1 / 365), ("hour", 1 / (365 * 24))):
    s1 = S * np.exp((0.08 - vol ** 2 / 2) * dt_ + vol * math.sqrt(dt_) * rng.standard_normal(20_000))
    un = np.array([black_scholes(s, K, T - dt_, r, vol) for s in s1]) - g0["price"]
    he = un - g0["delta"] * (s1 - S) - r * (g0["price"] - g0["delta"] * S) * dt_
    left[label] = float(he.std() / un.std())
out["answer"] = {"left_day": left["day"], "left_hour": left["hour"], "shrink": left["day"] / left["hour"]}
out["answer"]

## 3. Three ways to get the Greeks

**Delta** $\partial V/\partial S$, **gamma** $\partial^2 V/\partial S^2$, **theta** $\partial V/\partial t$ (calendar time; negative when the option loses value as time passes). By the formula (`bs_greeks`), by nudging the inputs, and by QuantLib's analytic engine.

In [ ]:
today = ql.Date(29, 9, 2026)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
process = ql.BlackScholesMertonProcess(
    ql.QuoteHandle(ql.SimpleQuote(S)), flat(0.0), flat(r),
    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), vol, dc)))
expiry = today + 365   # exactly one year on ACT/365F
g = bs_greeks(S, K, T, r, vol)
h, ht = 0.01, 1 / 365
bump = {"delta": (black_scholes(S + h, K, T, r, vol) - black_scholes(S - h, K, T, r, vol)) / (2 * h),
        "gamma": (black_scholes(S + 0.5, K, T, r, vol) - 2 * g["price"] + black_scholes(S - 0.5, K, T, r, vol)) / 0.25,
        "theta": (black_scholes(S, K, T - ht, r, vol) - g["price"]) / ht}          # one day passes
opt = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, K), ql.EuropeanExercise(expiry))
opt.setPricingEngine(ql.AnalyticEuropeanEngine(process))
qlg = {"delta": opt.delta(), "gamma": opt.gamma(), "theta": opt.theta()}
rows = [{"greek": name, "formula": g[name], "nudge": bump[name], "quantlib": qlg[name]} for name in ("delta", "gamma", "theta")]
for row in rows:
    assert abs(row["formula"] - row["quantlib"]) < 1e-10
    assert abs(row["nudge"] / row["formula"] - 1) < 0.01
out["greeks"] = {**g, "rows": rows, "theta_day": g["theta"] / 365, "theta_day_abs": abs(g["theta"]) / 365, "theta_day_cents": abs(g["theta"]) / 365 * 100,
                 "theta_year_abs": abs(g["theta"])}
pd.DataFrame(rows)

## 4. Delta across strikes

For our share at \$100: a deep in-the-money call (low strike) moves almost dollar for dollar with the share; a far out-of-the-money call hardly moves. With a month left instead of a year, the change from 0 to 1 is much sharper.

In [ ]:
strikes = np.arange(60.0, 150.01, 1.0)
delta_1y = [bs_greeks(S, k, T, r, vol)["delta"] for k in strikes]
delta_1m = [bs_greeks(S, k, 1 / 12, r, vol)["delta"] for k in strikes]
marks = [80.0, 105.0, 130.0]
out["delta"] = {"strikes": strikes.tolist(), "one_year": delta_1y, "one_month": delta_1m,
                "points_x": marks, "points_y": [bs_greeks(S, k, T, r, vol)["delta"] for k in marks],
                "k80": bs_greeks(S, 80, T, r, vol)["delta"], "k105": g["delta"], "k130": bs_greeks(S, 130, T, r, vol)["delta"],
                "ticks": [{"x": float(k), "label": f"${k}"} for k in (60, 75, 90, 105, 120, 135, 150)]}
{k: v for k, v in out["delta"].items() if not isinstance(v, list)}

## 5. Gamma

Gamma is how fast delta changes as the share moves. For a \$100-strike call it peaks near the money, and it grows sharply as expiry approaches: near expiry, the hedge flips quickly between almost no shares and a full share.

In [ ]:
prices = np.arange(60.0, 150.01, 1.0)
gamma_1y = [bs_greeks(s, 100.0, 1.0, r, vol)["gamma"] for s in prices]
gamma_1m = [bs_greeks(s, 100.0, 1 / 12, r, vol)["gamma"] for s in prices]
out["gamma"] = {"strike": 100.0, "prices": prices.tolist(), "one_year": gamma_1y, "one_month": gamma_1m,
                "atm_1y": bs_greeks(S, 100.0, 1.0, r, vol)["gamma"], "atm_1m": bs_greeks(S, 100.0, 1 / 12, r, vol)["gamma"],
                "atm_1w": bs_greeks(S, 100.0, 1 / 52, r, vol)["gamma"], "ours": g["gamma"],
                "ticks": [{"x": float(k), "label": f"${k}"} for k in (60, 75, 90, 100, 120, 135, 150)]}
out["gamma"]["ratio_1w"] = out["gamma"]["atm_1w"] / out["gamma"]["atm_1y"]
{k: v for k, v in out["gamma"].items() if not isinstance(v, list)}

## 6. Theta and the trade-off

Theta for our call is about $-\$5.57$ a year, $-1.5$ cents a day. The Black-Scholes equation (Episode 14), written with the Greeks, is an identity between them:

$$\Theta + \tfrac12\sigma^2S^2\,\Gamma + rS\,\Delta - rV = 0$$

The biggest pair is theta and the gamma term: a holder of gamma pays for it with theta.

In [ ]:
terms = {"theta": g["theta"], "gamma_term": 0.5 * vol ** 2 * S ** 2 * g["gamma"], "delta_term": r * S * g["delta"], "rate_term": -r * g["price"]}
terms["residual"] = sum(terms.values())
assert abs(terms["residual"]) < 1e-10
out["identity"] = {**terms, "theta_abs": abs(terms["theta"]), "rate_abs": abs(terms["rate_term"])}
out["identity"]

## 7. The break-even move

Own the call and hedge it with delta shares. Over one day the book's P&L is about $\tfrac12\Gamma(\Delta S)^2 + \Theta\,\Delta t$ plus small interest terms: a gain from any move, a loss from time. The two balance when $|\Delta S| \approx \sigma S \sqrt{\Delta t} = \$1.05$: move more than that and the hedged holder makes money; move less and they lose. Exact one-day repricing:

In [ ]:
dt = 1 / 365
moves = np.arange(-3.0, 3.001, 0.1)
cash = g["price"] - g["delta"] * S                           # negative: borrowed to buy the shares
pnl = [black_scholes(S + m, K, T - dt, r, vol) - g["price"] - g["delta"] * m - r * cash * dt for m in moves]
breakeven = vol * S * math.sqrt(dt)
out["breakeven"] = {"move": breakeven, "moves": moves.round(2).tolist(), "pnl": pnl,
                    "flat_day": pnl[len(moves) // 2], "flat_day_abs": abs(pnl[len(moves) // 2]),
                    "ticks": [{"x": float(x), "label": f"{'+' if x > 0 else ('−' if x < 0 else '')}${abs(x):.0f}"} for x in (-3, -2, -1, 0, 1, 2, 3)]}
signs = np.sign(pnl)
cross = moves[1:][np.diff(signs) != 0]
assert all(abs(abs(c) - breakeven) < 0.1 for c in cross)
out["breakeven"]["crossings"] = cross.tolist()
{k: v for k, v in out["breakeven"].items() if not isinstance(v, list)}

## 8. This episode's question

Who wants a quiet market, the option buyer or the seller? The seller: a hedged seller is short gamma and collects theta, so they gain on days the share moves less than about \$1.05 and lose on days it moves more.

In [ ]:
out["question"] = {"answer": "seller"}

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")